# 1. Snowflake setup and export to Ossie

This notebook builds the Snowflake side of the demo and exports the semantic view
to an Apache Ossie file on a stage. You download that file and upload it to
Databricks.

Order of work:
1. Set your database and schema.
2. Create the stage and load two tables from CSV.
3. Create a semantic view over those tables.
4. Export the semantic view to Ossie YAML on the stage.

You need the two CSV files (`customers.csv`, `orders.csv`) from the demo folder,
and a role that can create objects in the database and schema you choose. No admin
role is required.

## Step 1 - Set your database and schema

Run this cell first. The SQL cells below reference `{{DATABASE}}` and `{{SCHEMA}}` from here, so you change the target in one place.

In [ ]:
# Set the database and schema to work in. Pick a database and schema where your
# current role can create tables, a stage, and a semantic view. No admin role is
# needed; the notebook runs with your role's privileges. Use these same names in
# the Databricks notebook so the table references in the Ossie file line up.
DATABASE = "DEMOS"
SCHEMA   = "SEMANTIC_INTEROP"
print(f"Working in {DATABASE}.{SCHEMA}")

Create the schema if it does not exist, then set it as the working schema. If you do not have the privilege to create a schema, point `DATABASE` and `SCHEMA` at one that already exists and skip the create.

In [ ]:
CREATE SCHEMA IF NOT EXISTS {{DATABASE}}.{{SCHEMA}};

In [ ]:
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - Create the stage, then upload the CSVs

Run the cell to create the stage. Then upload `customers.csv` and `orders.csv` into
it before continuing:

- Snowsight: Data, Databases, your database, your schema, Stages, INTEROP_STAGE, then the + Files button.
- CLI: `snow stage copy customers.csv @<database>.<schema>.INTEROP_STAGE` (repeat for orders.csv).

The LIST cell confirms both files landed.

In [ ]:
CREATE STAGE IF NOT EXISTS {{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE
  COMMENT = 'Interop demo: CSV data and Ossie files';

In [ ]:
LIST @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE;

## Step 3 - Create the tables

In [ ]:
CREATE OR REPLACE TABLE {{DATABASE}}.{{SCHEMA}}.CUSTOMERS (
    customer_id    INT,
    customer_name  STRING,
    region         STRING
);

In [ ]:
CREATE OR REPLACE TABLE {{DATABASE}}.{{SCHEMA}}.ORDERS (
    order_id       INT,
    customer_id    INT,
    order_amount   INT,
    order_qty      INT
);

## Step 4 - Load the tables from the CSVs

Expected result: EAST 750/5/12, WEST 700/5/11.

In [ ]:
COPY INTO {{DATABASE}}.{{SCHEMA}}.CUSTOMERS
  FROM @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE/customers.csv
  FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
  ON_ERROR = ABORT_STATEMENT;

In [ ]:
COPY INTO {{DATABASE}}.{{SCHEMA}}.ORDERS
  FROM @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE/orders.csv
  FILE_FORMAT = (TYPE = CSV SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"')
  ON_ERROR = ABORT_STATEMENT;

In [ ]:
SELECT c.region, SUM(o.order_amount) AS total_amount, COUNT(o.order_id) AS order_count, SUM(o.order_qty) AS total_qty
FROM {{DATABASE}}.{{SCHEMA}}.ORDERS o
JOIN {{DATABASE}}.{{SCHEMA}}.CUSTOMERS c USING (customer_id)
GROUP BY c.region ORDER BY c.region;

## Step 5 - Create the semantic view

The view defines two metrics (`TOTAL_ORDER_AMOUNT`, `ORDER_COUNT`), a region
dimension, and the ORDERS-to-CUSTOMERS relationship. These are the definitions that
travel to Databricks.

In [ ]:
CREATE OR REPLACE SEMANTIC VIEW {{DATABASE}}.{{SCHEMA}}.SALES_SV
  TABLES (
    orders AS {{DATABASE}}.{{SCHEMA}}.ORDERS PRIMARY KEY (order_id),
    customers AS {{DATABASE}}.{{SCHEMA}}.CUSTOMERS PRIMARY KEY (customer_id)
  )
  RELATIONSHIPS (
    orders_to_customers AS orders (customer_id) REFERENCES customers (customer_id)
  )
  FACTS (
    orders.order_amount AS order_amount,
    orders.order_qty AS order_qty
  )
  DIMENSIONS (
    customers.region AS region,
    customers.customer_name AS customer_name
  )
  METRICS (
    orders.total_order_amount AS SUM(orders.order_amount),
    orders.order_count AS COUNT(orders.order_id)
  )
  COMMENT = 'Sales star for Ossie interop demo';

In [ ]:
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.SALES_SV
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;

### See the semantic view in Cortex Analyst

We built this view with SQL so the demo repeats cleanly, but it is an ordinary
semantic view, the same kind your teams build in the UI. Before moving on, open it
in Snowsight and look at it the way an analyst would:

- Find it under Data, your database, your schema, Semantic Views, or select it in
  Cortex Analyst under AI & ML.
- Review the tables, the region dimension, and the two metrics.
- Ask a question in plain language, for example "total order amount by region", and
  confirm the answer matches the numbers above.

Nothing here is specific to the interop demo. What crosses to Databricks in the next
steps is this same definition.

## Step 6 - Export the semantic view to Ossie on the stage

`SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW` returns the vendor-neutral Ossie model
as text (spec version 0.1.1). The COPY writes it to one file on the stage. The file
format uses no field or record delimiter, so the whole document lands as one value
instead of being split into rows and columns.

In [ ]:
COPY INTO @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE/ossie_from_snowflake.yaml
FROM (SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW('{{DATABASE}}.{{SCHEMA}}.SALES_SV'))
FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE COMPRESSION = NONE)
SINGLE = TRUE OVERWRITE = TRUE;

In [ ]:
LIST @{{DATABASE}}.{{SCHEMA}}.INTEROP_STAGE;

## Step 7 - Download the Ossie file

Download `ossie_from_snowflake.yaml` from the stage (Snowsight stage browser, or
`snow stage copy @<database>.<schema>.INTEROP_STAGE/ossie_from_snowflake.yaml ./`).

Next: open notebook 2 in Databricks and upload this file plus the two CSVs into the
notebook's folder. Use the same database and schema names in Databricks.